# LoRA vs QLoRA Fine-Tuning of Qwen2.5-1.5B-Instruct on Agricultural Q&A

# **1. Environment Setup**

In [5]:
!pip uninstall -y torchao

!pip install -q \
transformers==4.56.1 \
peft==0.17.1 \
trl==0.21.0 \
accelerate \
bitsandbytes \
datasets \
evaluate \
bert_score \
sentencepiece

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 95.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.9/511.9 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 46.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.1 MB/s eta 0:00:00


In [6]:
import transformers
import peft
import trl
import torch

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)

Torch: 2.10.0+cu128
Transformers: 4.56.1
PEFT: 0.17.1
TRL: 0.21.0


## 2. GPU Verification

In [7]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
    print("bf16 supported:", torch.cuda.is_bf16_supported())
else:
    print("WARNING: no GPU detected. Enable a GPU accelerator in Kaggle notebook settings.")

CUDA available: True
Device: Tesla T4
VRAM (GB): 15.6
bf16 supported: True


## 3. Reproducibility

In [8]:
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

## 4. Dataset Loading

In [9]:
from datasets import load_dataset

raw = load_dataset("KisanVaani/agriculture-qa-english-only", split="train")
print(raw)
print(raw[0])

Dataset({
    features: ['question', 'answers'],
    num_rows: 22615
})
{'question': 'why is crop rotation important in farming?', 'answers': 'This helps to prevent soil erosion and depletion, and can also help to control pests and diseases'}


In [10]:
import pandas as pd

df = raw.to_pandas()
print(df.shape)
print(df.isna().sum())
df.sample(5, random_state=SEED)

(22615, 2)
question    0
answers     0
dtype: int64


,question,answers
9334,name any one example of forest pest.,bark beetles
20612,How often can i sprinkle liquid manure?,"Generally, liquid manure can be applied every ..."
7144,how are plants harvested?,by hand or machine
18763,what are some of the advantages of Band placem...,"By placing the fertilizers in rows, you can sp..."
6492,what is rotenone?,Rotenone is a plant extract found in some spec...


## 5. Dataset Preprocessing (clean, filter, leakage-safe split)

In [11]:
def clean(example):
    return {
        "question": str(example["question"]).strip(),
        "answer": str(example["answers"]).strip(),
    }

cleaned = raw.map(clean)
cleaned = cleaned.filter(
    lambda ex: len(ex["question"]) > 5 and len(ex["answer"]) > 5
               and ex["question"].lower() != ex["answer"].lower()
)
cleaned = cleaned.shuffle(seed=SEED)

TRAIN_N, VAL_N, TEST_N = 4000, 500, 500
train_ds = cleaned.select(range(0, TRAIN_N))
val_ds   = cleaned.select(range(TRAIN_N, TRAIN_N + VAL_N))
test_ds  = cleaned.select(range(TRAIN_N + VAL_N, TRAIN_N + VAL_N + TEST_N))

print(len(train_ds), len(val_ds), len(test_ds))

Map:   0%|          | 0/22615 [00:00<?, ? examples/s]

Filter:   0%|          | 0/22615 [00:00<?, ? examples/s]

4000 500 500


## 6. Prompt / Instruction Formatting

In [15]:
SYSTEM = ("You are an agricultural assistant. Answer the farmer's question "
          "clearly, correctly, and concisely, in plain English.")

def to_train_text(example):
    text = (f"<|im_start|>system\n{SYSTEM}<|im_end|>\n"
            f"<|im_start|>user\n{example['question']}<|im_end|>\n"
            f"<|im_start|>assistant\n{example['answer']}<|im_end|>")
    return {"text": text}

def to_prompt_only(question):
    return (f"<|im_start|>system\n{SYSTEM}<|im_end|>\n"
            f"<|im_start|>user\n{question}<|im_end|>\n"
            f"<|im_start|>assistant\n")

train_sft = train_ds.map(to_train_text)
val_sft   = val_ds.map(to_train_text)
print(train_sft[0]["text"])

<|im_start|>system
You are an agricultural assistant. Answer the farmer's question clearly, correctly, and concisely, in plain English.<|im_end|>
<|im_start|>user
what is seed dormancy<|im_end|>
<|im_start|>assistant
Seed dormancy is the condition in which a seed fails to germinate even under conditions that are normally favorable for germination<|im_end|>


## 7. Base Model

In [17]:
pip install -U transformers huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 98.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 104.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 73.4 MB/s eta 0:00:00:00:01
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.7.0
    Uninstalling safetensors-0.7.0:
      Successfully uninstalled safetensors-0.7.0
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.4.3
    Uninstalling hf-xet-1.4.3:
      Successfully uninstalled hf-xet-1.4.3
  Attempting uninstall: click
    Found existing installation: click 8.3.3
    Uninstalling click-8.3.3:
      Successfully uninstalled click-8.3.3
  Attempting uninstall: hugging

In [19]:
!pip install "google-cloud-bigquery-storage>=2.0.0"

INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.0/308.0 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 98.8 MB/s eta 0:00:00:00:01
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
  Attempting uninstall: grpcio
    Found existing installation: grpcio 1.81.1
    Uninstalling grpcio-1.81.1:
      Successfully uninstalled grpcio-1.81.1
  Attempting uninstall: grpcio-status
    Found existing installation: grpcio-status 1.71.2
    Uninstalling grpcio-status-1.71.2:
      Successfully uninstalled grpcio-status-1.71.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installe

In [21]:
tokenizer.chat_template

NameError: name 'tokenizer' is not defined

In [20]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto",
)
print("Total params (M):", sum(p.numel() for p in base_model.parameters()) / 1e6)


RemoteEntryNotFoundError: 404 Client Error. (Request ID: Root=1-6ab6638b-33a58feb12c9a62e5e839847;29b9f191-2c67-407d-a349-818a1cb86c31)

Entry Not Found for url: https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct/tree/main/additional_chat_templates?recursive=false&expand=false.
additional_chat_templates does not exist on "main"

## 8. Baseline (before any fine-tuning)

In [23]:
@torch.no_grad()
def generate(model, question, max_new_tokens=200):
    prompt = to_prompt_only(question)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=False,
        repetition_penalty=1.1, pad_token_id=tokenizer.pad_token_id,
    )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

for ex in test_ds.select(range(3)):
    print("Q:", ex["question"])
    print("Base model:", generate(base_model, ex["question"]))
    print("Reference :", ex["answer"])
    print("---")

Q: what do i look when inspect the packaging.


NameError: name 'base_model' is not defined

## 9. Tokenization Check

In [22]:
lengths = [len(tokenizer(t)["input_ids"]) for t in train_sft.select(range(200))["text"]]
print("mean/median/95th pct token length:", np.mean(lengths), np.median(lengths), np.percentile(lengths, 95))
MAX_SEQ_LEN = 512  # chosen from the 95th percentile check above

NameError: name 'tokenizer' is not defined

## 10. LoRA Configuration

In [12]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

lora_model = get_peft_model(base_model, lora_config)
lora_model.print_trainable_parameters()

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


## 11. QLoRA Configuration

In [13]:
from transformers import BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)



qlora_base = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto",torch_dtype=torch.bfloat16)

# FIX: gradient checkpointing must be enabled ONCE, with consistent kwargs,
# before wrapping with LoRA. Previously this was enabled here with default
# kwargs, then re-enabled a second time inside SFTConfig with different
# kwargs (use_reentrant=False) -- that mismatch is what silently broke
# gradient flow into the adapters (see the training cell below for the
# full explanation and the eval-loss evidence).
qlora_base.config.use_cache = False
qlora_base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
qlora_base = prepare_model_for_kbit_training(
    qlora_base,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

qlora_lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM", target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
qlora_model = get_peft_model(qlora_base, qlora_lora_config)

# THE ACTUAL FIX: with use_reentrant=False, PEFT's prepare_model_for_kbit_training
# deliberately SKIPS calling enable_input_require_grads() (it assumes
# non-reentrant checkpointing doesn't need it). That assumption breaks with
# this double-enable ordering, so we call it explicitly ourselves, after
# get_peft_model, as a safety net. This one line restores gradient flow
# from the frozen 4-bit embeddings into the trainable LoRA adapters.
qlora_model.enable_input_require_grads()

qlora_model.print_trainable_parameters()


trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


In [14]:
sample = tokenizer(train_sft[0]["text"], return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN).to(qlora_model.device)
sample["labels"] = sample["input_ids"].clone()

qlora_model.train()
out = qlora_model(**sample)
out.loss.backward()

# NOTE: we check lora_B, not lora_A. LoRA's B matrix is zero-initialized by
# design (so the adapter is a no-op at step 0), which means d(loss)/d(A) is
# MATHEMATICALLY ALWAYS 0.0 at initialization, regardless of whether the
# setup is healthy. lora_B's gradient is the meaningful signal to check here.
lora_b_param = next(p for n, p in qlora_model.named_parameters() if "lora_B" in n and p.requires_grad)
print("Loss:", out.loss.item())
print("Sample LoRA-B param requires_grad:", lora_b_param.requires_grad)
print("Sample LoRA-B param grad is None:", lora_b_param.grad is None)
if lora_b_param.grad is not None:
    print("Sample LoRA-B param grad abs sum:", lora_b_param.grad.abs().sum().item())

qlora_model.zero_grad()

# Expect: grad is NOT None, and grad abs sum is a small positive number (not 0.0).


Loss: 3.694704055786133
Sample LoRA-B param requires_grad: True
Sample LoRA-B param grad is None: False
Sample LoRA-B param grad abs sum: 40.58937454223633


## 12. Training Configuration + LoRA Training

In [16]:
from trl import SFTConfig, SFTTrainer

lora_sft_config = SFTConfig(
    output_dir="outputs/lora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    optim="adamw_torch",
    report_to="none",
    dataset_text_field="text",
    max_length=MAX_SEQ_LEN,
    packing=False,
    seed=SEED,
)

lora_trainer = SFTTrainer(
    model=lora_model,
    args=lora_sft_config,
    train_dataset=train_sft,
    eval_dataset=val_sft,
    processing_class=tokenizer,
)



Adding EOS to train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

In [19]:
lora_trainer.train()
lora_trainer.save_model("outputs/lora")
tokenizer.save_pretrained("outputs/lora")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
50,1.059900,1.059799
100,0.992800,1.001140
150,1.010400,0.958488
200,0.919900,0.914507
250,0.885800,0.873441
300,0.773000,0.830684
350,0.778900,0.790715
400,0.752500,0.754308
450,0.700000,0.727682
500,0.723300,0.703254


('outputs/lora/tokenizer_config.json',
 'outputs/lora/special_tokens_map.json',
 'outputs/lora/chat_template.jinja',
 'outputs/lora/vocab.json',
 'outputs/lora/merges.txt',
 'outputs/lora/added_tokens.json',
 'outputs/lora/tokenizer.json')

## 13. QLoRA Training

In [ ]:
# ============================================================
# QLoRA SFT — FIXED TRAINING CONFIG + TRAINING LOOP
# ============================================================

qlora_sft_config = SFTConfig(
    output_dir="outputs/qlora",

    # -------------------------
    # Training duration
    # -------------------------
    num_train_epochs=3,

    # -------------------------
    # Batch / memory efficiency
    # -------------------------
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    # Effective batch size = 2 x 8 = 16

    # -------------------------
    # Learning rate
    # -------------------------
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,

    # -------------------------
    # Regularization
    # -------------------------
    weight_decay=0.01,
    max_grad_norm=1.0,

    # -------------------------
    # Precision
    # -------------------------
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),

    # -------------------------
    # Optimizer
    # -------------------------
    optim="paged_adamw_8bit",

    # -------------------------
    # Sequence length
    # -------------------------
    max_length=MAX_SEQ_LEN,
    packing=False,

    # -------------------------
    # Evaluation
    # -------------------------
    eval_strategy="steps",
    eval_steps=25,

    # -------------------------
    # Logging
    # -------------------------
    logging_strategy="steps",
    logging_steps=5,

    # -------------------------
    # Checkpoints
    # -------------------------
    save_strategy="steps",
    save_steps=25,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # -------------------------
    # Dataset
    # -------------------------
    dataset_text_field="text",

    # -------------------------
    # Reproducibility
    # -------------------------
    seed=SEED,

    # -------------------------
    # Disable external logging
    # -------------------------
    report_to="none",

    # FIX: gradient_checkpointing is intentionally NOT set here.
    # It's already enabled directly on qlora_base in the config cell above,
    # with matching use_reentrant=False kwargs. Setting it again here made
    # SFTTrainer call model.gradient_checkpointing_enable() a SECOND time
    # with its own kwargs -- that double-enable is what silently broke
    # gradient flow into the LoRA adapters (eval loss was frozen at
    # 3.745506 for all 20 evaluations in the previous run). Leaving this
    # unset means Trainer won't touch the checkpointing state we already
    # configured correctly.

    # -------------------------
    # Keep unused columns
    # -------------------------
    remove_unused_columns=False,
)


# ============================================================
# CREATE TRAINER
# ============================================================

qlora_trainer = SFTTrainer(
    model=qlora_model,
    args=qlora_sft_config,
    train_dataset=train_sft,
    eval_dataset=val_sft,
    processing_class=tokenizer,
)

# ============================================================
# THE FIX: SFTTrainer.__init__, on seeing that `model` is already a
# PeftModel, internally re-runs peft.prepare_model_for_kbit_training() on
# it (trl/trainer/sft_trainer.py -> _prepare_peft_model -> is_qlora branch).
# That function's very first line freezes ALL parameters, including the
# LoRA adapters we already correctly marked trainable in the config cell
# above. Because peft_config=None was passed here, nothing re-enables them
# afterward. We restore it explicitly, before .train() is called.
# ============================================================
n_restored = 0
for name, param in qlora_trainer.model.named_parameters():
    if "lora_" in name:
        param.requires_grad = True
        n_restored += 1

print(f"Restored requires_grad=True on {n_restored} LoRA parameter tensors")
qlora_trainer.model.print_trainable_parameters()
# Expect the SAME trainable count/% printed in the QLoRA Configuration cell
# above (4,358,144 / 0.2815%). If this now prints 0 trainable parameters,
# something upstream changed and this fix needs re-checking.


# ============================================================
# TRAIN
# ============================================================

print("=" * 60)
print("STARTING QLoRA FINE-TUNING")
print("=" * 60)

print(f"Training examples   : {len(train_sft)}")
print(f"Validation examples : {len(val_sft)}")
print(f"Epochs              : {qlora_sft_config.num_train_epochs}")
print(f"Micro batch size    : {qlora_sft_config.per_device_train_batch_size}")
print(f"Gradient accumulation: {qlora_sft_config.gradient_accumulation_steps}")
print(f"Learning rate       : {qlora_sft_config.learning_rate}")
print(f"Max sequence length : {qlora_sft_config.max_length}")

print("=" * 60)


train_result = qlora_trainer.train()


# ============================================================
# TRAINING SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)

print(f"Training loss : {train_result.training_loss:.4f}")

if torch.cuda.is_available():
    print(
        f"Peak GPU memory: "
        f"{torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
    )


# ============================================================
# FINAL VALIDATION
# ============================================================

print("\n" + "=" * 60)
print("FINAL VALIDATION")
print("=" * 60)

eval_results = qlora_trainer.evaluate()

for key, value in eval_results.items():
    if isinstance(value, (int, float)):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")


# ============================================================
# SAVE BEST MODEL / ADAPTER
# ============================================================

print("\n" + "=" * 60)
print("SAVING MODEL")
print("=" * 60)

qlora_trainer.save_model("outputs/qlora")
tokenizer.save_pretrained("outputs/qlora")

print("Saved to: outputs/qlora")
print("=" * 60)


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Restored requires_grad=True on 224 LoRA parameter tensors
trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815
STARTING QLoRA FINE-TUNING
Training examples   : 4000
Validation examples : 500
Epochs              : 3
Micro batch size    : 2
Gradient accumulation: 8
Learning rate       : 0.0001
Max sequence length : 512


Step,Training Loss,Validation Loss
25,2.680400,2.396640
50,1.337900,1.298566
75,1.108700,1.076931
100,1.039100,1.045408
125,1.071100,1.028100
150,1.079300,1.014017
175,1.042100,0.997970
200,0.976700,0.984161
225,0.974200,0.974618
250,0.972500,0.960805


In [34]:
print(type(qlora_model))

print("lm_head:",
      qlora_model.base_model.model.lm_head.weight.dtype)

for name, module in qlora_model.named_modules():
    if "lm_head" in name:
        print(name, module.weight.dtype)

<class 'peft.peft_model.PeftModelForCausalLM'>
lm_head: torch.bfloat16
base_model.model.lm_head torch.bfloat16


> Im having session ending problem so i trained models first downloaded the weights locally and importing from the zip files and continued the work this frees me from the problem of session ending and also RAM problem 

In [4]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/models/mallikarjunjadi/lora-qlora/pytorch/default/1/lora_final/adapter_model.safetensors
/kaggle/input/models/mallikarjunjadi/lora-qlora/pytorch/default/1/lora_final/merges.txt
/kaggle/input/models/mallikarjunjadi/lora-qlora/pytorch/default/1/lora_final/adapter_config.json
/kaggle/input/models/mallikarjunjadi/lora-qlora/pytorch/default/1/lora_final/README.md
/kaggle/input/models/mallikarjunjadi/lora-qlora/pytorch/default/1/lora_final/tokenizer.json
/kaggle/input/models/mallikarjunjadi/lora-qlora/pytorch/default/1/lora_final/vocab.json
/kaggle/input/models/mallikarjunjadi/lora-qlora/pytorch/default/1/lora_final/tokenizer_config.json
/kaggle/input/models/mallikarjunjadi/lora-qlora/pytorch/default/1/lora_final/chat_template.jinja
/kaggle/input/models/mallikarjunjadi/lora-qlora/pytorch/default/1/lora_final/special_tokens_map.json
/kaggle/input/models/mallikarjunjadi/lora-qlora/pytorch/default/1/lora_final/added_tokens.json
/kaggle/input/models/mallikarjunjadi/lora-qlora/pytor

In [7]:
!find /kaggle/input -name "adapter_model.safetensors"

/kaggle/input/models/mallikarjunjadi/lora-qlora/pytorch/default/1/lora_final/adapter_model.safetensors
/kaggle/input/models/mallikarjunjadi/lora-qlora/pytorch/default/1/qlora_final/adapter_model.safetensors


In [6]:
!find /kaggle/input -type f | head -100

/kaggle/input/models/mallikarjunjadi/lora-qlora/pytorch/default/1/lora_final/adapter_model.safetensors
/kaggle/input/models/mallikarjunjadi/lora-qlora/pytorch/default/1/lora_final/merges.txt
/kaggle/input/models/mallikarjunjadi/lora-qlora/pytorch/default/1/lora_final/adapter_config.json
/kaggle/input/models/mallikarjunjadi/lora-qlora/pytorch/default/1/lora_final/README.md
/kaggle/input/models/mallikarjunjadi/lora-qlora/pytorch/default/1/lora_final/tokenizer.json
/kaggle/input/models/mallikarjunjadi/lora-qlora/pytorch/default/1/lora_final/vocab.json
/kaggle/input/models/mallikarjunjadi/lora-qlora/pytorch/default/1/lora_final/tokenizer_config.json
/kaggle/input/models/mallikarjunjadi/lora-qlora/pytorch/default/1/lora_final/chat_template.jinja
/kaggle/input/models/mallikarjunjadi/lora-qlora/pytorch/default/1/lora_final/special_tokens_map.json
/kaggle/input/models/mallikarjunjadi/lora-qlora/pytorch/default/1/lora_final/added_tokens.json
/kaggle/input/models/mallikarjunjadi/lora-qlora/pytor

In [25]:
from peft import PeftModel

lora_model = PeftModel.from_pretrained(
    base_model,
    "/kaggle/input/models/mallikarjunjadi/lora-qlora/pytorch/default/1/lora_final"
)

In [20]:
q = "What is machine learning?"

print(generate(lora_model, q))

Machine learning is a type of artificial intelligence that allows machines to learn without being explicitly programmed


In [23]:
qlora_model = PeftModel.from_pretrained(
    base_model,
    "/kaggle/input/models/mallikarjunjadi/lora-qlora/pytorch/default/1/qlora_final"
)


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [24]:
print(generate(qlora_model, "What is machine learning?"))

Machine learning is a type of artificial intelligence that allows computer programs to improve automatically through experience


In [28]:
!find /kaggle/input -name "trainer_state.json"

## 14. Training Curves

In [26]:
import matplotlib.pyplot as plt

def plot_curves(trainer, title):
    history = trainer.state.log_history
    train_loss = [(h["step"], h["loss"]) for h in history if "loss" in h]
    eval_loss = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]
    plt.figure()
    if train_loss:
        plt.plot(*zip(*train_loss), label="train loss")
    if eval_loss:
        plt.plot(*zip(*eval_loss), label="eval loss")
    plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.title(title)
    plt.show()

plot_curves(lora_trainer, "LoRA training curve")
plot_curves(qlora_trainer, "QLoRA training curve")

NameError: name 'lora_trainer' is not defined

In [24]:
import shutil

shutil.make_archive(
    "/kaggle/working/outputs_backup",
    "zip",
    "/kaggle/working/outputs"
)

print("ZIP created: /kaggle/working/outputs_backup.zip")

ZIP created: /kaggle/working/outputs_backup.zip


In [25]:
# ============================================================
# 1. SAVE LoRA + QLoRA TO KAGGLE WORKING DIRECTORY
# ============================================================

import os

lora_path = "/kaggle/working/lora_final"
qlora_path = "/kaggle/working/qlora_final"

os.makedirs(lora_path, exist_ok=True)
os.makedirs(qlora_path, exist_ok=True)

# LoRA
lora_model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)

# QLoRA
qlora_model.save_pretrained(qlora_path)
tokenizer.save_pretrained(qlora_path)

print("LoRA:")
print(os.listdir(lora_path))

print("\nQLoRA:")
print(os.listdir(qlora_path))

LoRA:
['added_tokens.json', 'special_tokens_map.json', 'README.md', 'tokenizer_config.json', 'chat_template.jinja', 'tokenizer.json', 'merges.txt', 'vocab.json', 'adapter_model.safetensors', 'adapter_config.json']

QLoRA:
['added_tokens.json', 'special_tokens_map.json', 'README.md', 'tokenizer_config.json', 'chat_template.jinja', 'tokenizer.json', 'merges.txt', 'vocab.json', 'adapter_model.safetensors', 'adapter_config.json']


In [26]:
import shutil

shutil.make_archive(
    "/kaggle/working/lora_final",
    "zip",
    "/kaggle/working/lora_final"
)

shutil.make_archive(
    "/kaggle/working/qlora_final",
    "zip",
    "/kaggle/working/qlora_final"
)

print("ZIP files created.")

ZIP files created.


## 15. Evaluation (ROUGE-L, BERTScore, Exact Match)

In [33]:
base_scores = evaluate_model(base_model, eval_test)
print(base_scores)

{'rouge_l': np.float64(0.2551480781582922), 'bertscore_f1': 0.8941554963588715, 'exact_match_rate': 0.0}


In [37]:
def generate(model, prompt):
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False
        )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )


In [38]:
base_for_lora = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)

lora_model = PeftModel.from_pretrained(
    base_for_lora,
    "/kaggle/input/models/mallikarjunjadi/lora-qlora/pytorch/default/1/lora_final"
)

lora_results = evaluate_model(
    lora_model,
    eval_test
)

print(lora_results)

#del lora_model
#del base_for_lora
#gc.collect()
#torch.cuda.empty_cache()

{'rouge_l': np.float64(0.1313042859809044), 'bertscore_f1': 0.8381205558776855, 'exact_match_rate': 0.0}


In [39]:
base_for_qlora = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16,
    device_map="auto"
)

qlora_model = PeftModel.from_pretrained(
    base_for_qlora,
    "/kaggle/input/models/mallikarjunjadi/lora-qlora/pytorch/default/1/qlora_final"
)

qlora_results = evaluate_model(
    qlora_model,
    eval_test
)

print(qlora_results)

#del qlora_model
#del base_for_qlora
#gc.collect()
#torch.cuda.empty_cache()

{'rouge_l': np.float64(0.12612685774152627), 'bertscore_f1': 0.843788492679596, 'exact_match_rate': 0.0}


## 16. Base vs Fine-Tuned Qualitative Comparison

In [1]:
rows = []
for ex in test_ds.select(range(10)):
    rows.append({
        "question": ex["question"],
        "base_model": generate(base_model, ex["question"]),
        "lora": generate(lora_model, ex["question"]),
        "qlora": generate(qlora_model, ex["question"]),
        "reference": ex["answer"],
    })
comparison_df = pd.DataFrame(rows)
comparison_df


NameError: name 'test_ds' is not defined

## 17. LoRA vs QLoRA Resource Comparison

In [30]:
import subprocess

print("LoRA adapter size:")
print(subprocess.run(["du", "-sh", "outputs/lora"], capture_output=True, text=True).stdout)
print("QLoRA adapter size:")
print(subprocess.run(["du", "-sh", "outputs/qlora"], capture_output=True, text=True).stdout)

# Peak memory per run should be captured live during Sections 12-13 with
# torch.cuda.reset_peak_memory_stats() before training and
# torch.cuda.max_memory_allocated() after — record those numbers in results/fine_tuned_results.md.


LoRA adapter size:
163M	outputs/lora

QLoRA adapter size:
114M	outputs/qlora



## 18. Ablation Experiment: LoRA Rank

In [31]:
ablation_results = {}
for r in [4, 16, 64]:
    cfg_r = LoraConfig(
        r=r, lora_alpha=r * 2, lora_dropout=0.05, bias="none",
        task_type="CAUSAL_LM", target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )
    fresh_base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16, device_map="auto"
    )
    model_r = get_peft_model(fresh_base, cfg_r)

    ablation_config = SFTConfig(
        output_dir=f"outputs/ablation_r{r}",
        num_train_epochs=1,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        logging_steps=10,
        eval_strategy="no",
        save_strategy="no",
        bf16=torch.cuda.is_bf16_supported(),
        report_to="none",
        dataset_text_field="text",
        max_length=MAX_SEQ_LEN,
        packing=False,
        seed=SEED,
    )
    trainer_r = SFTTrainer(model=model_r, args=ablation_config, train_dataset=train_sft, processing_class=tokenizer)
    trainer_r.train()
    ablation_results[r] = evaluate_model(model_r, test_ds.select(range(100)))

print(ablation_results)


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


RuntimeError: The size of tensor a (4) must match the size of tensor b (8) at non-singleton dimension 0

## 19. Failure Analysis (Required)

In [ ]:
flagged = []
for ex in test_ds.select(range(50)):
    pred = generate(qlora_model, ex["question"])
    if len(pred) < 15 or len(pred) > 600:
        flagged.append({"question": ex["question"], "prediction": pred, "reference": ex["answer"]})

print(f"{len(flagged)} flagged examples out of 50 for manual review")
for f in flagged[:10]:
    print(f)


## 20. Final Conclusions

Fill this cell in only after Sections 15-19 have produced real numbers.
Answer each research question from the introduction using the measured
ROUGE-L/BERTScore/exact-match values, the memory/time/adapter-size
comparison, the rank-ablation results, and the documented failures.
Do not write conclusions here before the experiment has actually run —
copy the final numbers into `results/baseline_results.md` and
`results/fine_tuned_results.md` at the same time.
